# 01 画像の基礎 — 画像は「数値の配列」

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 解説（この文章）→ コード → 結果（画像）→ ✅ チェックポイント、の順で進みます。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

**目標**：「画像とは何か」を**コンピュータの言葉（数値の配列）**で理解し、OpenCV で画像を**読み・形を変え・色を変え・保存する**という、全ての画像処理の土台になる操作を手で動かす。

> **🎯 このノートのゴール**
>
> - [ ] 画像が `(高さ, 幅, 3)` の numpy 配列であることを `shape` で確認できる
> - [ ] OpenCV の色順が **BGR** であることを理解する
> - [ ] リサイズ・トリミング・描画ができる
> - [ ] グレースケール／HSV 変換で「色で物を選ぶ」ができる

## 1-0. 画像は「数値の格子」
デジタル画像は、小さな点（**画素／ピクセル**）を格子状に並べたもの。カラー画像では 1 画素が **3 つの数値**（色の成分）を持ち、各成分は **0–255** の整数（8bit）です。

| 用語 | 意味 | 例 |
|---|---|---|
| **画素 (pixel)** | 画像を構成する最小の点 | 1 画素 = (B,G,R) の 3 値 |
| **解像度** | 幅 × 高さ の画素数 | 640×480、4056×3040 |
| **チャンネル** | 色の成分の数 | カラー=3(B,G,R)、グレー=1 |
| **ビット深度** | 1 成分の段階数 | 8bit = 0–255 の 256 段階 |

> **🟥 OpenCV の色順は「BGR」**：多くのソフトは **RGB**（赤・緑・青）ですが、**OpenCV は BGR**（青・緑・赤）。
> これを忘れると**赤と青が入れ替わった変な色**になります。OpenCV の中だけなら気にせず使えますが、
> **matplotlib や PIL（RGB 前提）に渡すときだけ** `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` で変換します。

## 1-1. 題材の画像を用意
まずカメラで 1 枚撮って、変数 `img` に入れます（以降このノートで使い回します）。

In [ ]:
import cv2, numpy as np, time
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'  # 日本語ラベルの豆腐対策（要 fonts-noto-cjk）
%matplotlib inline
from picamera2 import Picamera2

def show(bgr, title='', size=(6,4), gray=False):
    '''OpenCV の BGR 画像をノートにインライン表示するヘルパー'''
    plt.figure(figsize=size)
    if gray:
        plt.imshow(bgr, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    plt.axis('off');
    if title: plt.title(title)
    plt.show()

picam = Picamera2()
picam.configure(picam.create_preview_configuration(main={'size':(640,480),'format':'RGB888'}))
picam.start(); time.sleep(1)
img = picam.capture_array(); picam.close()
show(img, '題材の画像 img')

## 1-2. 画像の正体を見る
画像は `(高さ, 幅, 3)` の **numpy 配列**。1 画素は 3 つの数値（各 0–255）です。

> **`img[y, x]` の順番に注意**：配列アクセスは `img[行, 列]` = `img[y, x]`（**縦が先**）。
> 座標の `(x, y)` とは逆で、`shape` も `(高さ, 幅, 3)` と**高さ（行数）が先**に来ます。

In [ ]:
print('shape:', img.shape)        # (高さ, 幅, チャンネル数)
print('dtype:', img.dtype)        # uint8 = 0..255
print('左上の画素 (B,G,R):', img[0, 0])

✅ **チェックポイント**：`shape: (480, 640, 3)` のように表示されれば OK。

> **大事なクセ**：OpenCV の色順は **BGR**（青・緑・赤）。次でその意味を目で確かめます。

## 1-3. BGR を体感する
OpenCV の配列を **RGB と思って**表示すると、赤と青が入れ替わって見えます。

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(img); plt.axis('off')
plt.title('間違い：BGR配列をそのまま表示（色が変）')
plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off')
plt.title('正解：BGR→RGB に変換して表示')
plt.show()

→ 左が「色が変」になります。**matplotlib や PIL に渡すときだけ** `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` を通すと覚えましょう。

## 1-4. リサイズ・トリミング・描画
大きい画像は重いので、**小さくして**から処理するのが定石。トリミング（切り抜き）は**配列のスライス**でできます。

> **サイズ指定は `(幅, 高さ)`、`shape` は `(高さ, 幅)`**：`cv2.resize` に渡すサイズは **(幅, 高さ)** の順、
> 一方 `img.shape` は **(高さ, 幅, 3)** の順。OpenCV は「**サイズは (w,h)、配列は (h,w)**」という二重基準です。

In [ ]:
small = cv2.resize(img, (320, 240))         # サイズは (幅, 高さ)
roi   = img[100:300, 200:440]                # スライスで切り抜き [y1:y2, x1:x2]
canvas = img.copy()
cv2.rectangle(canvas, (200,100), (440,300), (0,255,0), 2)  # 緑の枠 (色は BGR)
cv2.putText(canvas, 'ROI', (200,90), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
show(roi, '切り抜き roi'); show(canvas, '枠と文字を描画')

## 1-5. 色空間：グレースケールと HSV
目的に応じて**色の表し方（色空間）を変える**と、後の処理が一気に簡単になります。

| 色空間 | 何が嬉しいか | 使いどころ |
|---|---|---|
| **グレースケール** | 色を捨て明るさだけ(1ch)。処理が軽い | 二値化・エッジ・輪郭の前段 |
| **HSV** | 色相 H・彩度 S・明度 V に分ける。**色で物を選びやすい** | 特定の色（赤い物 等）の抽出 |

> **OpenCV の H（色相）は 0–179**：一般に色相は 0–360°ですが、OpenCV では 8bit に収めるため **半分の 0–179**。
> 「赤」は H が 0 付近と 179 付近の**両端**に分かれるので、赤を抜くときは 2 つの範囲を足す、と覚えておくと後で困りません。

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
show(gray, 'グレースケール（明るさだけ）', gray=True)

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, (35,80,80), (85,255,255))   # 緑のあたり (H は 0..179)
show(mask, '緑を白で抜き出したマスク', gray=True)
print('緑と判定した画素数:', int((mask>0).sum()))

✅ **チェックポイント**：グレー画像と、緑だけ白いマスク画像が表示されれば OK。

## 1-6. やってみよう
- `inRange` の範囲を **青**（H を 100〜130）に変えて、青い物だけ抜き出してみる
- 切り抜く範囲 `[y1:y2, x1:x2]` を変えて、好きな場所を切り出す

> 🤖 **ChatGPT に聞いてみよう**
> 「OpenCV(Python)で、HSV を使って**青い物だけ**白く抜き出すコードを書いて。OpenCV の H は 0–179 に注意」と聞いて、出たコードをこの下のセルで試そう。

In [ ]:
# ここに自分のコードを書いて Shift+Enter

## ✅ 到達確認

- [ ] 画像が `(高さ, 幅, 3)` の `uint8` 配列だと説明できる
- [ ] OpenCV は BGR、H は 0–179 という「クセ」を押さえた
- [ ] 縮小・トリミング・描画・色空間変換を自分で実行できた

---
次は **02_フィルタと輪郭** で、画像の中の**物体を数え**ます。